In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/PietroGhersetich/RecSys_challenge_2025

Cloning into 'RecSys_challenge_2025'...
remote: Enumerating objects: 1499, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 1499 (delta 93), reused 0 (delta 0), pack-reused 1359 (from 1)
Receiving objects: 100% (1499/1499), 626.68 MiB | 39.97 MiB/s, done.
Resolving deltas: 100% (921/921), done.
Updating files: 100% (222/222), done.


In [2]:
! pip install PyGithub requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.9 MB/s eta 0:00:00


In [3]:
! cd /kaggle/working/RecSys_challenge_2025 && python run_compile_all_cython.py

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
In file included from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from MatrixFactorizationImpressions_Cython_Epoch.c:1252:
/usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
 

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
import tqdm
import gc
import time
import os
import shutil
import optuna
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import scipy.sparse as sps

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [5]:
from github import Github
from github import Auth

auth = Auth.Token(token)
g = Github(auth=auth)

repo = None
for r in g.get_user().get_repos():
    if r.name == 'RecSys_challenge_2025':
        repo = r
        print('Repository found')

Repository found


In [6]:
def upload_file(filepath_kaggle, filepath_github, commit_message):
    try:
        contents = repo.get_contents(filepath_github)
        with open(filepath_kaggle, "rb") as file:
            repo.update_file(contents.path, commit_message, file.read(), contents.sha)
    except Exception:
        with open(filepath_kaggle, "rb") as file:
            repo.create_file(filepath_github, commit_message, file.read())

In [7]:
db_name = 'knn_study.db'
json_name_item = 'item_knn_best_params.json'
json_name_user = 'user_knn_best_params.json'

config = {
    'tune_parameters_item_knn': True,
    'tune_parameters_user_knn': True,
    'database_name': db_name,
    'database_path': f'/kaggle/working/{db_name}',
    'save_json': True,
    'json_name_item': json_name_item,
    'json_name_user': json_name_user,
    'copy_prev_tuning_db': True,
    'train_best_model': False,
    'copy_prev_best_params': True,
    'save_github': True,
    'candidates': '20',
    'n_trials_item': 1000, # 1000 -> 5h
    'n_trials_user': 300  # 300 -> 5h
}

In [8]:
try:
    shutil.copyfile(
        f'/kaggle/working/RecSys_challenge_2025/tuning_results/recall@{config["candidates"]}/{config["database_name"]}', 
        config['database_path']
    )
except FileNotFoundError:
    pass # if not present optuna will create it

# Data Import

In [9]:
train_data = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
user_ids_test = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

In [10]:
data = np.ones(len(train_data))

URM_all = sps.csr_matrix((data, (train_data['row'].values, train_data['col'].values)),
                         shape=(train_data['row'].max() + 1, train_data['col'].max() + 1))

URM_all

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3043058 stored elements and shape (27095, 6969)>

In [11]:
%cd /kaggle/working/RecSys_challenge_2025

/kaggle/working/RecSys_challenge_2025


In [12]:
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

from Evaluation.Evaluator import EvaluatorHoldout

np.int = int
np.random.seed(42)
np.bool = bool
np.float = float

In [13]:
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

In [14]:
import sqlite3

def print_all_rows(table_name, database_path):
    connection = sqlite3.connect(database_path)
    cursor = connection.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        print(f"All rows from table '{table_name}':")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"Error reading data from table '{table_name}': {e}")

    finally:
        cursor.close()
        connection.close()

print_all_rows('studies', config['database_path'])

All rows from table 'studies':
(1, 'item_knn_Optimization')
(2, 'user_knn_Optimization')


In [15]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_itemknn(optuna_trial):
    recommender_instance = ItemKNNCFRecommender(URM_train)
    
    # 1. Choose the similarity metric first
    similarity = optuna_trial.suggest_categorical("similarity", 
        ['cosine', 'asymmetric', 'dice', 'jaccard', 'tanimoto', 'tversky', 'euclidean'])
    
    # 2. Global parameters
    params = {
        'similarity': similarity,
        'topK': optuna_trial.suggest_int("topK", 10, 1000),
        'shrink': optuna_trial.suggest_int("shrink", 0, 500),
        'normalize': optuna_trial.suggest_categorical("normalize", [True, False]),
        'feature_weighting': optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"]),
    }
    
    # 3. Conditional parameters based on similarity
    if similarity == 'asymmetric':
        params["asymmetric_alpha"] = optuna_trial.suggest_float("asymmetric_alpha", 0, 2.0, log=False)
        params["normalize"] = True 
        
    elif similarity == 'tversky':
        params['tversky_alpha'] = optuna_trial.suggest_float("tversky_alpha", 0, 2.0, log=False)
        params['tversky_beta'] = optuna_trial.suggest_float("tversky_beta", 0, 2.0, log=False)
        params["normalize"] = True 
        
    elif similarity == 'euclidean':
        params["normalize_avg_row"] = optuna_trial.suggest_categorical("normalize_avg_row", [True, False])
        params["similarity_from_distance_mode"] = optuna_trial.suggest_categorical("similarity_from_distance_mode", ["lin", "log", "exp"])
        params["normalize"] = optuna_trial.suggest_categorical("normalize", [True, False])

    recommender_instance.fit(**params)
    
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [16]:
%%time
# an Itemknn study_name MUST contain "item_knn"
if config['tune_parameters_item_knn']:
    study = optuna.create_study(direction='maximize', study_name='item_knn_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_itemknn, n_trials=config['n_trials_item'])

[I 2025-12-29 22:34:46,727] Using an existing study with name 'item_knn_Optimization' instead of creating a new one.


Previous trials found: 1306
Similarity column 6969 (100.0%), 1665.49 column/sec. Elapsed time 4.18 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 16.71 sec. Users per second: 1620


[I 2025-12-29 22:35:09,450] Trial 1306 finished with value: 0.23015826450262541 and parameters: {'similarity': 'tversky', 'topK': 61, 'shrink': 84, 'normalize': True, 'feature_weighting': 'TF-IDF', 'tversky_alpha': 0.18922594264782217, 'tversky_beta': 1.9936754183612981}. Best is trial 1282 with value: 0.2522216197519622.


Similarity column 6969 (100.0%), 1687.88 column/sec. Elapsed time 4.13 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 16.25 sec. Users per second: 1665


[I 2025-12-29 22:35:30,652] Trial 1307 finished with value: 0.23563647459112333 and parameters: {'similarity': 'tversky', 'topK': 44, 'shrink': 91, 'normalize': True, 'feature_weighting': 'TF-IDF', 'tversky_alpha': 0.18828188172245455, 'tversky_beta': 1.9429642864156882}. Best is trial 1282 with value: 0.2522216197519622.


...

Similarity column 6969 (100.0%), 1642.23 column/sec. Elapsed time 4.24 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 16.06 sec. Users per second: 1686


[I 2025-12-30 04:33:16,074] Trial 2304 finished with value: 0.22722682732166097 and parameters: {'similarity': 'tversky', 'topK': 78, 'shrink': 25, 'normalize': True, 'feature_weighting': 'TF-IDF', 'tversky_alpha': 0.16093690141049533, 'tversky_beta': 1.9417719818208976}. Best is trial 2220 with value: 0.25264184591005884.


Similarity column 6969 (100.0%), 1672.06 column/sec. Elapsed time 4.17 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 17.68 sec. Users per second: 1531


[I 2025-12-30 04:33:38,725] Trial 2305 finished with value: 0.20654217168841174 and parameters: {'similarity': 'tanimoto', 'topK': 56, 'shrink': 38, 'normalize': True, 'feature_weighting': 'TF-IDF'}. Best is trial 2220 with value: 0.25264184591005884.


CPU times: user 5h 53min 49s, sys: 5min 4s, total: 5h 58min 53s
Wall time: 5h 58min 54s


In [17]:
if config['save_json'] and config['save_github'] and config['tune_parameters_item_knn']:
    j_name = config['json_name_item']
    l_path = f"/kaggle/working/{j_name}"
    r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
    
    best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
    
    try:
        for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
            if "item_knn" not in s.study_name: 
                continue

            if s.best_trial and s.best_trial.value > best_val:
                best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
    except:
        best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name

    prev_val, exists = -float('inf'), False
    try:
        prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
        exists = True
    except:
        pass

    if not exists or best_val > prev_val:
        print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
        with open(l_path, 'w') as f:
            json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
        if config.get('save_github'):
            upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
    else:
        print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

Updating JSON. Best: 0.252642 (Prev: 0.252222)


In [18]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_userknn(optuna_trial):
    recommender_instance = UserKNNCFRecommender(URM_train)
    
    # 1. Choose the similarity metric first
    similarity = optuna_trial.suggest_categorical("similarity", 
        ['cosine', 'asymmetric', 'dice', 'jaccard', 'tanimoto', 'tversky', 'euclidean'])
    
    # 2. Global parameters
    params = {
        'similarity': similarity,
        'topK': optuna_trial.suggest_int("topK", 10, 1000),
        'shrink': optuna_trial.suggest_int("shrink", 0, 500),
        'normalize': optuna_trial.suggest_categorical("normalize", [True, False]),
        'feature_weighting': optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"]),
    }
    
    # 3. Conditional parameters based on similarity
    if similarity == 'asymmetric':
        params["asymmetric_alpha"] = optuna_trial.suggest_float("asymmetric_alpha", 0, 2.0, log=False)
        params["normalize"] = True 
        
    elif similarity == 'tversky':
        params['tversky_alpha'] = optuna_trial.suggest_float("tversky_alpha", 0, 2.0, log=False)
        params['tversky_beta'] = optuna_trial.suggest_float("tversky_beta", 0, 2.0, log=False)
        params["normalize"] = True 
        
    elif similarity == 'euclidean':
        params["normalize_avg_row"] = optuna_trial.suggest_categorical("normalize_avg_row", [True, False])
        params["similarity_from_distance_mode"] = optuna_trial.suggest_categorical("similarity_from_distance_mode", ["lin", "log", "exp"])
        params["normalize"] = optuna_trial.suggest_categorical("normalize", [True, False])

    recommender_instance.fit(**params)
    
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [19]:
%%time
# an Userknn study_name MUST contain "user_knn"
if config['tune_parameters_user_knn']:
    study = optuna.create_study(direction='maximize', study_name='user_knn_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_userknn, n_trials=config['n_trials_user'])

[I 2025-12-30 04:33:43,206] Using an existing study with name 'user_knn_Optimization' instead of creating a new one.


Previous trials found: 605
Similarity column 27095 (100.0%), 610.01 column/sec. Elapsed time 44.42 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 22.43 sec. Users per second: 1207


[I 2025-12-30 04:34:51,459] Trial 605 finished with value: 0.18106897474945696 and parameters: {'similarity': 'jaccard', 'topK': 430, 'shrink': 241, 'normalize': False, 'feature_weighting': 'TF-IDF'}. Best is trial 521 with value: 0.22516357135725665.


Similarity column 27095 (100.0%), 605.98 column/sec. Elapsed time 44.71 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 24.50 sec. Users per second: 1105


[I 2025-12-30 04:36:01,498] Trial 606 finished with value: 0.22462737471687624 and parameters: {'similarity': 'asymmetric', 'topK': 374, 'shrink': 258, 'normalize': False, 'feature_weighting': 'TF-IDF', 'asymmetric_alpha': 0.29636661940668796}. Best is trial 521 with value: 0.22516357135725665.


...

Similarity column 27095 (100.0%), 598.87 column/sec. Elapsed time 45.24 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 25.14 sec. Users per second: 1077


[I 2025-12-30 10:24:46,498] Trial 903 finished with value: 0.2238444261307233 and parameters: {'similarity': 'asymmetric', 'topK': 391, 'shrink': 195, 'normalize': False, 'feature_weighting': 'TF-IDF', 'asymmetric_alpha': 0.10574714548342806}. Best is trial 521 with value: 0.22516357135725665.


Similarity column 27095 (100.0%), 593.70 column/sec. Elapsed time 45.64 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 20.55 sec. Users per second: 1317


[I 2025-12-30 10:25:53,932] Trial 904 finished with value: 0.11763428609583826 and parameters: {'similarity': 'tversky', 'topK': 430, 'shrink': 256, 'normalize': False, 'feature_weighting': 'TF-IDF', 'tversky_alpha': 0.9648343379556319, 'tversky_beta': 0.1509308074525667}. Best is trial 521 with value: 0.22516357135725665.


CPU times: user 5h 49min 3s, sys: 2min 59s, total: 5h 52min 2s
Wall time: 5h 52min 10s


In [20]:
if config['save_json'] and config['save_github'] and config['tune_parameters_user_knn']:
    j_name = config['json_name_user']
    l_path = f"/kaggle/working/{j_name}"
    r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
    
    best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
    
    try:
        for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
            if "user_knn" not in s.study_name: 
                continue

            if s.best_trial and s.best_trial.value > best_val:
                best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
    except:
        best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name

    prev_val, exists = -float('inf'), False
    try:
        prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
        exists = True
    except:
        pass

    if not exists or best_val > prev_val:
        print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
        with open(l_path, 'w') as f:
            json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
        if config.get('save_github'):
            upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
    else:
        print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

No update needed. Best: 0.225164 <= Saved: 0.225164


In [21]:
if config['save_github'] and (config['tune_parameters_user_knn'] or config['tune_parameters_item_knn']):
    upload_file(
        config['database_path'], 
        f'tuning_results/recall@{config["candidates"]}/{config["database_name"]}',
        'Tuning db updated results (from kaggle notebook)'
    )